# Amulet Engine — YOLOv11 Training Pipeline
**ก่อนกดรัน:** ตรวจสอบเมนูด้านบน **Runtime -> Change runtime type -> เลือก `T4 GPU`** แล้วกด Save

In [1]:
!pip install -q ultralytics roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="RJu6VkIYQ5QiQYCfwnvA")
project = rf.workspace("al-by-ball").project("dual-view_amulet_ai")
version = project.version(1)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Dual-View_Amulet_AI-1 in yolov11:: 100%|██████████| 16546/16546 [00:02<00:00, 5942.24it/s] 


In [2]:
import os
import glob
import yaml

TARGET_CLASSES = [
    "somdej_watrakang_front", "somdej_watrakang_back",
    "somdej_bangkhunprom_front", "somdej_bangkhunprom_back",
    "somdej_ketchaiyo_front", "somdej_ketchaiyo_back",
    "phra_nangphaya_front", "phra_nangphaya_back",
    "phra_phong_suphan_front", "phra_phong_suphan_back",
    "phra_rod_front", "phra_rod_back",
    "phra_sumkor_front", "phra_sumkor_back",
    "defect"
]

yaml_path = f"{dataset.location}/data.yaml"
with open(yaml_path, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

old_names = data_cfg["names"]
old_id_to_name = {int(k): v for k, v in old_names.items()} if isinstance(old_names, dict) else {i: n for i, n in enumerate(old_names)}
old_to_new_id = {old_id: TARGET_CLASSES.index(name) for old_id, name in old_id_to_name.items() if name in TARGET_CLASSES}

kept_images, removed_images = 0, 0

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset.location, split, "images")
    lbl_dir = os.path.join(dataset.location, split, "labels")
    if not os.path.exists(img_dir):
        continue

    for img_path in glob.glob(os.path.join(img_dir, "*")):
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(lbl_dir, f"{base_name}.txt")

        new_lines = []
        if os.path.exists(lbl_path):
            with open(lbl_path, "r") as lf:
                for line in lf:
                    parts = line.strip().split()
                    if parts and int(parts[0]) in old_to_new_id:
                        parts[0] = str(old_to_new_id[int(parts[0])])
                        new_lines.append(" ".join(parts))

        if new_lines:
            with open(lbl_path, "w") as lf:
                lf.write("\n".join(new_lines) + "\n")
            kept_images += 1
        else:
            os.remove(img_path)
            if os.path.exists(lbl_path):
                os.remove(lbl_path)
            removed_images += 1

data_cfg["nc"] = len(TARGET_CLASSES)
data_cfg["names"] = TARGET_CLASSES
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_cfg, f, allow_unicode=True)

print(f"พร้อมเทรน! เหลือภาพที่มีป้ายเป้าหมาย {kept_images} ภาพ | ตัดภาพว่าง/นอกขอบเขตออก {removed_images} ภาพ")

พร้อมเทรน! เหลือภาพที่มีป้ายเป้าหมาย 3018 ภาพ | ตัดภาพว่าง/นอกขอบเขตออก 5253 ภาพ


In [3]:
import os
from ultralytics import YOLO
from google.colab import files

model = YOLO("yolo11s.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16
)

best_weight_path = os.path.join(results.save_dir, "weights", "best.pt")
files.download(best_weight_path)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dual-View_Amulet_AI-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, frac

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>